# Session 2: Building Your First Agent — From Framework to Harness

**Data Science Dojo x SambaNova | Deep Agents Webinar Series**

In Session 1, we learned what makes coding agents work — the 5 Pillars. Today we build one from scratch, and learn a critical vocabulary: **runtimes, frameworks, and harnesses**.

By the end of this notebook, you'll:
1. Write the raw ReAct loop from scratch (no framework)
2. See how LangGraph wraps it with production features
3. Run the same task under **3 different harness configurations** and compare results
4. Understand why harness engineering matters more than model choice

## Section 0: Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(os.path.join("..", "..", ".env"), override=True)

%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.insert(0, os.path.join(".."))

from utils import format_messages

### Langfuse Tracing (Optional)

If you have [Langfuse](https://langfuse.com) credentials, we'll enable tracing so you can inspect every LLM call and tool invocation in the Langfuse dashboard. If not, the notebook works fine without it.

In [ ]:
from langfuse.langchain import CallbackHandler

langfuse_handler = None
if os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY"):
    langfuse_handler = CallbackHandler()
    print("Langfuse tracing enabled — traces will appear in your Langfuse dashboard.")
else:
    print("Langfuse keys not found — tracing disabled. Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to enable.")

### LangSmith Tracing (Optional)

If you have [LangSmith](https://smith.langchain.com) credentials set in your `.env`, tracing is automatic — every LLM call and tool invocation will appear in your LangSmith dashboard. If not, the notebook works fine without it.

In [ ]:
from langsmith import traceable

langsmith_enabled = os.environ.get("LANGSMITH_TRACING") == "true" and os.environ.get("LANGSMITH_API_KEY")
if langsmith_enabled:
    print(f"LangSmith tracing enabled — project: {os.environ.get('LANGSMITH_PROJECT', 'default')}")
else:
    print("LangSmith not configured — tracing disabled. Set LANGSMITH_TRACING=true and LANGSMITH_API_KEY to enable.")

In [ ]:
def get_config():
    """Build run config with tracing callbacks when available."""
    callbacks = []
    if langfuse_handler:
        callbacks.append(langfuse_handler)
    if callbacks:
        return {"callbacks": callbacks}
    return {}

In [ ]:
from langchain_sambanova import ChatSambaNova

# We'll use the same model throughout — only the harness changes
MODEL = ChatSambaNova(model="MiniMax-M2.5", temperature=0.0)

# Our benchmark task — used across all 3 configurations
BENCHMARK_TASK = (
    "Research the current state of nuclear fusion energy. Cover: "
    "(1) the key players and companies involved, "
    "(2) the most significant recent breakthroughs, and "
    "(3) realistic timelines to commercial viability. "
    "Produce a well-structured report with sources."
)

---
## Section 1: The ReAct Loop from Scratch

In Session 1, we jumped straight to `create_react_agent`. Today we peel back the abstraction and build the **raw loop** ourselves.

The ReAct pattern is simple:
1. **Reason** — the LLM thinks about what to do
2. **Act** — the LLM calls a tool
3. **Observe** — we feed the tool result back
4. **Repeat** until the LLM decides it's done (no more tool calls)

That's it. It's a **while loop**.

In [ ]:
from typing import Literal, Union
from langchain_core.tools import tool


@tool
def calculator(
    operation: Literal["add", "subtract", "multiply", "divide"],
    a: Union[int, float],
    b: Union[int, float],
) -> Union[int, float]:
    """A simple two-input calculator.

    Args:
        operation: The math operation to perform
        a: First number
        b: Second number
    """
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b if b != 0 else "Error: division by zero"

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage


def react_loop(model, tools: list, user_query: str, max_iterations: int = 10):
    """A ReAct agent built from scratch — no framework, just a while loop."""

    # Bind tools to the model so it knows what's available
    model_with_tools = model.bind_tools(tools)

    # Build a lookup dict: tool name -> tool function
    tool_map = {t.name: t for t in tools}

    # Start with the user's message
    messages = [HumanMessage(content=user_query)]

    for i in range(max_iterations):
        # REASON: Ask the LLM what to do
        response = model_with_tools.invoke(messages)
        messages.append(response)

        # If no tool calls, the agent is done
        if not response.tool_calls:
            break

        # ACT + OBSERVE: Execute each tool call and feed results back
        for tool_call in response.tool_calls:
            tool_fn = tool_map[tool_call["name"]]
            result = tool_fn.invoke(tool_call["args"])
            messages.append(
                ToolMessage(content=str(result), tool_call_id=tool_call["id"])
            )

    return messages

In [ ]:
# Run it!
messages = react_loop(
    MODEL,
    tools=[calculator],
    user_query="What is (3.1 * 4.2) + (10 / 3)? Show your work.",
)

format_messages(messages)

**That's the entire agent.** A while loop, tool dispatch, and message accumulation.

Every framework — LangGraph, CrewAI, OpenAI Agents SDK — wraps this same pattern. The difference is what they add on top.

---
## Section 2: ReAct with LangGraph

Now let's see what a **runtime** gives us. LangGraph wraps the same loop but adds:
- Checkpointing (resume from any state)
- Streaming (see thinking in real-time)
- Human-in-the-loop (pause for approval)
- Durable execution (survive crashes)

In [ ]:
from langgraph.prebuilt import create_react_agent
from IPython.display import Image, display

# Two lines — same loop, but with runtime features
agent = create_react_agent(
    model=MODEL,
    tools=[calculator],
)

# Visualize the graph
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
result = agent.invoke(
    {"messages": [("user", "What is (3.1 * 4.2) + (10 / 3)? Show your work.")]},
    config=get_config(),
)

format_messages(result["messages"])

Same loop. Same result. But now we have checkpointing, streaming, and observability for free.

**But this is still just an agent — not a harness.** It has no planning ability, no memory beyond messages, no way to persist its work. Let's fix that.

---
## Section 3: Runtimes, Frameworks, and Harnesses

Before we start building, we need a vocabulary for what we're building.

### The Three Layers

| Layer | What It Provides | Examples |
|-------|-----------------|----------|
| **Runtime** | Infrastructure: durable execution, checkpointing, streaming, state persistence | LangGraph, Temporal, Inngest |
| **Framework** | Abstractions: tool definitions, message protocols, model wrappers | LangChain, OpenAI Agents SDK, Google ADK |
| **Harness** | Complete system: default prompts, opinionated tools, planning, middleware | Claude Code, Cursor, LangChain deep-agents, **what we'll build today** |

### The Key Insight

Claude Code, Cursor, and LangChain's deep-agents aren't just "agents using a framework." They're **harnesses** — complete systems with opinions about how to solve problems. When you build a production agent, you're building a harness too.

### The Three Knobs of Harness Engineering

There are three levers you can pull to improve an agent **without changing the model**:

1. **System Prompts** — Instructions, persona, constraints, planning templates
2. **Tools & Capabilities** — File systems, search, planning tools, delegation
3. **Middleware** — Loop detection, self-verification, context assembly

> **The LangChain team improved from 52.8% to 66.5% on TerminalBench 2.0 purely through harness engineering — no model change.** Same model. Just better prompts, better tools, better middleware.

Let's prove this ourselves. We'll run the **same task** on the **same model** under 3 harness configurations and compare the results.

---
## Section 4: Experiment — Config A (Bare Agent Baseline)

**Harness knobs:** None. No system prompt, no state tools. Just the model + a search tool.

This is our baseline — how does the model perform when we give it nothing to work with?

In [ ]:
from langchain_community.tools import TavilySearchResults

# Basic search tool — the only capability we give the bare agent
tavily_search = TavilySearchResults(max_results=3)

In [ ]:
# Config A: Bare agent — no system prompt, no state tools
agent_config_a = create_react_agent(
    model=MODEL,
    tools=[tavily_search],
)

print("Running Config A (bare agent) on benchmark task...")
result_a = agent_config_a.invoke(
    {"messages": [("user", BENCHMARK_TASK)]},
    config=get_config(),
)

print(f"\n--- Config A Results ---")
print(f"Total messages: {len(result_a['messages'])}")
print(f"Tool calls: {sum(1 for m in result_a['messages'] if hasattr(m, 'tool_calls') and m.tool_calls)}")
print(f"Has todos: {'todos' in result_a}")
print(f"Has files: {'files' in result_a}")

In [ ]:
# Show the full message trace
format_messages(result_a["messages"])

In [ ]:
# Save the final answer for comparison
config_a_answer = result_a["messages"][-1].content
config_a_msg_count = len(result_a["messages"])

**Observations:**
- No plan — the agent jumps straight into searching
- No structured output — just a stream of consciousness
- No persistent artifacts — once the context window closes, everything is lost
- The model did its best, but the harness gave it nothing to work with

---
## Section 5: Experiment — Config B (Add System Prompt)

**Harness knob turned: System Prompt (Knob 1)**

Same model, same tools. We just add **words** — a system prompt with planning instructions.

In [ ]:
PLANNING_PROMPT = """You are a research agent. Follow these steps for EVERY research task:

1. **Plan first**: Before searching, write out your research plan with specific questions to answer.
2. **Search systematically**: Use broad searches first, then narrow based on what you find.
3. **Reflect after each search**: After each search, assess: What did I learn? What's still missing?
4. **Structure your output**: Organize findings into clear sections with headers.
5. **Cite sources**: Include URLs or source names for all claims.
6. **Stop when sufficient**: Don't over-research. 3-5 searches is usually enough.

Think like a human researcher with limited time. Be systematic, not exhaustive.
"""

print("System prompt added. Zero code changes — just words.")

In [ ]:
# Config B: Same agent + system prompt
agent_config_b = create_react_agent(
    model=MODEL,
    tools=[tavily_search],
    prompt=PLANNING_PROMPT,
)

print("Running Config B (with system prompt) on benchmark task...")
result_b = agent_config_b.invoke(
    {"messages": [("user", BENCHMARK_TASK)]},
    config=get_config(),
)

print(f"\n--- Config B Results ---")
print(f"Total messages: {len(result_b['messages'])}")
print(f"Tool calls: {sum(1 for m in result_b['messages'] if hasattr(m, 'tool_calls') and m.tool_calls)}")
print(f"Has todos: {'todos' in result_b}")
print(f"Has files: {'files' in result_b}")

In [ ]:
format_messages(result_b["messages"])

In [ ]:
config_b_answer = result_b["messages"][-1].content
config_b_msg_count = len(result_b["messages"])

**Observations:**
- The agent now **plans before searching** (if it follows the prompt)
- Output is more structured — headers, sections, sources
- Still no persistent state — no todos, no files
- But already noticeably better. **Same model. Same tools. Just words.**

---
## Section 6: Experiment — Config C (Full Harness)

**Harness knobs turned: System Prompt (Knob 1) + Tools & State (Knob 2)**

Now we add the real power: custom state with TODO tracking and a virtual file system. The agent can plan explicitly (todos) and persist its research (files).

We'll build each piece from scratch so you understand how it works.

### Step 1: Custom State (DeepAgentState)

LangGraph's default `AgentState` only has a `messages` field. We extend it with:
- **`todos`**: A list of task items for planning
- **`files`**: A dictionary mapping filenames to content (our virtual file system)

In [ ]:
from typing import Annotated, Literal, NotRequired
from typing_extensions import TypedDict
from langgraph.prebuilt.chat_agent_executor import AgentState


class Todo(TypedDict):
    """A structured task item."""
    content: str
    status: Literal["pending", "in_progress", "completed"]


def file_reducer(left, right):
    """Merge two file dictionaries. Right side wins on conflicts."""
    if left is None:
        return right
    elif right is None:
        return left
    else:
        return {**left, **right}


class DeepAgentState(AgentState):
    """Extended agent state with task tracking and virtual file system."""
    todos: NotRequired[list[Todo]]
    files: Annotated[NotRequired[dict[str, str]], file_reducer]

### Deep Dive: Why `file_reducer` Matters

Without a reducer, state updates **replace** the entire field. With `file_reducer`, they **merge**.

Let's see the difference:

In [ ]:
# WITHOUT a reducer: replacement behavior
existing_files = {"report.md": "# My Report", "notes.md": "Some notes"}
new_update = {"data.csv": "col1,col2\n1,2"}

# Plain assignment would LOSE existing files
print("Without reducer (plain assignment):")
print(f"  Result: {new_update}")
print(f"  report.md and notes.md are GONE!\n")

# WITH file_reducer: merge behavior
print("With file_reducer:")
merged = file_reducer(existing_files, new_update)
print(f"  Result: {merged}")
print(f"  All files preserved, new file added!")

This one-line annotation — `Annotated[dict, file_reducer]` — prevents a whole class of bugs where tools overwrite each other's files.

### Step 2: The Command Pattern (Stateful Tools)

Normal tools return a string that the LLM sees. But our planning and file tools need to **modify state directly**. That's what LangGraph's `Command` type does.

Let's trace what happens step by step:

In [ ]:
from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command


@tool(parse_docstring=True)
def write_todos(
    todos: list[Todo],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Create or update the task list for planning and progress tracking.

    Args:
        todos: List of Todo items with content and status
        tool_call_id: Injected tool call identifier
    """
    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(
                    f"Updated todo list: {len(todos)} items",
                    tool_call_id=tool_call_id,
                )
            ],
        }
    )

### Deep Dive: Command vs Plain Return

What happens when a tool returns a **plain string** vs a **Command**?

**Normal tool (returns string):**
```
Tool returns "42"  
  -> LangGraph wraps it: ToolMessage(content="42")  
  -> Appended to state["messages"]  
  -> LLM sees it next turn  
  -> Only messages field changes
```

**Command tool (returns Command):**
```
Tool returns Command(update={"todos": [...], "messages": [ToolMessage(...)]})
  -> LangGraph applies ALL updates atomically:
     - state["todos"] = [...]
     - state["messages"].append(ToolMessage(...))
  -> Multiple state fields change at once
  -> LLM sees the ToolMessage, but todos are updated silently
```

Command is the bridge between **tools** (a harness concern) and **state** (a runtime concern).

In [ ]:
@tool(parse_docstring=True)
def read_todos(
    state: Annotated[dict, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> str:
    """Read the current TODO list to review progress.

    Args:
        state: Injected agent state
        tool_call_id: Injected tool call identifier
    """
    todos = state.get("todos", [])
    if not todos:
        return "No todos in the list."

    status_icons = {"pending": "[ ]", "in_progress": "[~]", "completed": "[x]"}
    lines = []
    for i, todo in enumerate(todos, 1):
        icon = status_icons.get(todo["status"], "[?]")
        lines.append(f"{i}. {icon} {todo['content']}")
    return "\n".join(lines)

### Step 3: Virtual File System Tools

In [ ]:
@tool
def ls(state: Annotated[dict, InjectedState]) -> list[str]:
    """List all files in the virtual filesystem."""
    return list(state.get("files", {}).keys())


@tool(parse_docstring=True)
def read_file(
    file_path: str,
    state: Annotated[dict, InjectedState],
) -> str:
    """Read a file from the virtual filesystem.

    Args:
        file_path: Path to the file to read
        state: Injected agent state
    """
    files = state.get("files", {})
    if file_path not in files:
        return f"Error: File '{file_path}' not found"
    content = files[file_path]
    lines = content.splitlines()
    numbered = [f"{i+1:4d}  {line}" for i, line in enumerate(lines)]
    return "\n".join(numbered)


@tool(parse_docstring=True)
def write_file(
    file_path: str,
    content: str,
    state: Annotated[dict, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Write content to a file in the virtual filesystem.

    Args:
        file_path: Path for the file
        content: Content to write
        state: Injected agent state
        tool_call_id: Injected tool call identifier
    """
    files = state.get("files", {})
    files[file_path] = content
    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(f"Wrote file: {file_path}", tool_call_id=tool_call_id)
            ],
        }
    )

### Step 4: Enhanced System Prompt (Harness-Aware)

Now we update the prompt to tell the agent about its new capabilities:

In [ ]:
HARNESS_PROMPT = """You are a research agent with planning and file management capabilities.

## Workflow
1. **Plan**: Use write_todos to create a research plan BEFORE searching.
2. **Research**: Search for information. After each search, IMMEDIATELY save findings to a file using write_file.
3. **Track**: After saving findings, use read_todos and update your todos to mark progress.
4. **Synthesize**: Once all research is done, use ls to see your files, read_file to review them, then write a final report file.
5. **Complete**: Mark all todos as completed when done.

## Rules
- Always create todos FIRST before any research
- Only one todo should be in_progress at a time
- ALWAYS use write_file to save search results — this is critical for context management
- After each search, save key findings to a file (e.g., "fusion_key_players.md")
- Use read_todos after completing a task to remind yourself of the plan
- Use ls to check what files you have already created
- Keep searches focused: 3-5 searches max
- Write a final report file at the end with all findings synthesized
"""

print("Harness prompt ready. Agent now knows about planning and file management.")

### Step 5: Build and Run Config C

In [ ]:
# Config C: Full harness — system prompt + state tools + DeepAgentState
agent_config_c = create_react_agent(
    model=MODEL,
    tools=[tavily_search, write_todos, read_todos, write_file, read_file, ls],
    state_schema=DeepAgentState,
    prompt=HARNESS_PROMPT,
)

# Visualize — notice the graph is the same structure, but the state and tools are richer
display(Image(agent_config_c.get_graph().draw_mermaid_png()))

In [ ]:
print("Running Config C (full harness) on benchmark task...")
result_c = agent_config_c.invoke(
    {"messages": [("user", BENCHMARK_TASK)]},
    config=get_config(),
)

print(f"\n--- Config C Results ---")
print(f"Total messages: {len(result_c['messages'])}")
print(f"Tool calls: {sum(1 for m in result_c['messages'] if hasattr(m, 'tool_calls') and m.tool_calls)}")
print(f"Has todos: {'todos' in result_c and bool(result_c.get('todos'))}")
print(f"Has files: {'files' in result_c and bool(result_c.get('files'))}")

In [ ]:
# Show the plan the agent made
print("=== Agent's Research Plan (Todos) ===")
for i, todo in enumerate(result_c.get("todos", []), 1):
    status_icons = {"pending": "[ ]", "in_progress": "[~]", "completed": "[x]"}
    icon = status_icons.get(todo["status"], "[?]")
    print(f"  {i}. {icon} {todo['content']}")

In [ ]:
# Show the files the agent created
print("=== Files Created ===")
for filename, content in result_c.get("files", {}).items():
    print(f"\n--- {filename} ({len(content)} chars) ---")
    # Show first 500 chars of each file
    print(content[:500])
    if len(content) > 500:
        print(f"  ... ({len(content) - 500} more chars)")

In [ ]:
# Full message trace
format_messages(result_c["messages"])

In [ ]:
config_c_answer = result_c["messages"][-1].content
config_c_msg_count = len(result_c["messages"])

---
## Section 7: Side-by-Side Comparison

Same model. Same task. Three harness configurations. Let's compare.

In [ ]:
from rich.table import Table
from rich.console import Console

console = Console()
table = Table(title="Harness Experiment: Same Model, Same Task, Different Configurations")

table.add_column("Metric", style="bold")
table.add_column("Config A\n(Bare Agent)", style="red")
table.add_column("Config B\n(+ Prompt)", style="yellow")
table.add_column("Config C\n(Full Harness)", style="green")

table.add_row(
    "System Prompt",
    "None",
    "Planning instructions",
    "Full harness prompt",
)
table.add_row(
    "Custom State",
    "No",
    "No",
    "DeepAgentState (todos + files)",
)
table.add_row(
    "Tools",
    "tavily_search only",
    "tavily_search only",
    "tavily + todos + files",
)
table.add_row(
    "Message Count",
    str(config_a_msg_count),
    str(config_b_msg_count),
    str(config_c_msg_count),
)
table.add_row(
    "Created Plan?",
    "No",
    "In-context only",
    f"Yes ({len(result_c.get('todos', []))} todos)",
)
table.add_row(
    "Persisted Files?",
    "No",
    "No",
    f"Yes ({len(result_c.get('files', {}))} files)",
)
table.add_row(
    "Answer Length",
    f"{len(config_a_answer)} chars",
    f"{len(config_b_answer)} chars",
    f"{len(config_c_answer)} chars",
)

console.print(table)

### What This Proves

The **same model** produced dramatically different results based solely on the **harness** around it:

- **Config A** (bare): No structure, no persistence, shallow results
- **Config B** (+ prompt): Better structure from instructions alone — the cheapest harness knob
- **Config C** (full harness): Plans explicitly, persists research to files, produces structured artifacts

> **This is harness engineering.** The model didn't get smarter — the system around it got better.

---
## Section 8: Knob 3 Preview — Middleware

We've used two knobs: **prompts** and **tools/state**. The third knob is **middleware** — hooks that run before or after model calls and tool executions.

Full middleware (loop detection, self-verification) is advanced territory we'll cover in Sessions 5-6. But here's a taste:

In [ ]:
# Simple middleware concept: a pre-tool hook that logs every tool call
tool_call_log = []

def logging_hook(tool_call):
    """Log every tool call for observability."""
    tool_call_log.append({
        "tool": tool_call["name"],
        "args_keys": list(tool_call["args"].keys()),
    })
    print(f"  [hook] Tool called: {tool_call['name']}")


# In a production harness, you'd also add:
# - Loop detection: if the agent calls the same tool with same args 3x, force a new strategy
# - Verification: after the agent says "done", run a check pass
# - Context assembly: before the first LLM call, inject environment info

print("Middleware is the most advanced harness knob.")
print("It modifies agent behavior without changing the agent code.")
print("We'll build real middleware in Sessions 5-6.")

---
## Section 9: What We've Started Building

Let's map what we built today to the three harness knobs:

| Knob | What We Built Today | What's Still Ahead |
|------|--------------------|-----------|
| **Prompts** | Planning workflow, search limits, TODO instructions | Context engineering, memory management (Session 3) |
| **Tools & State** | `write_todos`, `read_todos`, `write_file`, `read_file`, `ls`, `DeepAgentState` | Sub-agent delegation, MCP integration (Sessions 4-5) |
| **Middleware** | (Preview) Logging hooks | Loop detection, self-verification, context compression (Session 6) |

We've laid the foundation of the `deep-agents-from-scratch` harness — orchestration and basic state management. But a production harness needs much more: context engineering to manage long-running tasks, middleware to catch failure loops, sub-agents for context isolation, and evaluation to know if it's actually working. That's what the rest of this series builds.

### Key Takeaways

1. **ReAct is a while loop** — every framework wraps this same pattern
2. **Runtimes** (LangGraph) give you checkpointing, streaming, durability
3. **Frameworks** (LangChain) give you abstractions and tool definitions
4. **Harnesses** are complete agent systems — we've started building one with prompts, tools, and state
5. **Harness engineering > model swapping** — prompts, tools, and middleware move the needle more than changing models
6. **Command pattern** bridges tools and state — enabling stateful tools
7. **Reducers** (like `file_reducer`) prevent state overwrites in concurrent updates

---
## Exercises

### Exercise 1: Prompt Engineering
Modify ONLY the system prompt in Config B. Make it more specific about output format (e.g., require markdown headers, bullet points, a conclusion section). Re-run the benchmark task and compare the output to the original Config B.

### Exercise 2: Add a New Tool
Add an `edit_file` tool to Config C that does find-and-replace on existing files:
```python
@tool
def edit_file(file_path, old_text, new_text, state, tool_call_id) -> Command:
    # Read the file, replace old_text with new_text, write it back
    ...
```
Run the agent and see if it uses the edit tool vs. rewriting entire files.

### Exercise 3: Test Generalization
Run Config A vs Config C on a completely different task (e.g., "Compare Python, Rust, and Go for building web APIs"). Does the harness advantage hold across different task types?

---
## Resources

- [SambaNova](https://sambanova.ai/)
- [SambaNova Cloud](https://cloud.sambanova.ai/)
- [SambaNova Documentation](https://docs.sambanova.ai/)
- [Data Science Dojo](https://datasciencedojo.com/)


---
## Next Session: April 15 — Context Engineering

We introduced the three harness knobs today. Next session we go deep on **context assembly** — the most impactful knob for complex tasks:

- Virtual file systems for **context offloading** (move raw data out of messages)
- TODO lists for **planning and progress tracking** (prevent mission drift)
- Sub-agent spawning for **context quarantine** (isolate specialized tasks)

This is where agents go from good to great.